# Drone Chase Tag: self-play değerlendirmesi (GPU)

Bir self-play koşusunun bütün sürümleri için iki ölçüm (koşunun dosyaları repoda, eğitim gerekmez):
- **Elo turnuvası:** her kovalayan × her kaçan, çift başına 256 maç. Havuzun İÇİNDE ilerleme var mı, döngü var mı?
- **Unutma testi:** her sürüm SABİT elle yazılmış rakibe karşı (kovalayan vs OPEN, kaçan vs elle yazılmış kovalayan).
  Ajanlar birbirine karşı uzmanlaşırken dış rakiplere karşı kötüleşiyor mu? Gürültüsüz + olasılıklı.

1. **Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU** (yoksa L4) seç.
2. Hücreleri sırayla çalıştır. Sonuçlar Drive'a (`results/<koşu>/`) yazılır ve tek zip olarak iner.
3. Zip'i `D:\İndirilenler`e indir; `experiments/<koşu>/` altına konup commit'lenir.

**Bu notebook `eval-sp-faz1-anchor2-s0` koşusuna ait.** Kod `a5d92fd` sürümüne sabitli; sonuçlar için `experiments/LOG.md`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir("/content")                        # hücre nerede çalışırsa çalışsın, önce sabit bir yere geç
REPO = "https://github.com/HasaNSaygili/DroneChaseTag.git"
if os.path.isdir("/content/proje/.git"):    # daha önce indirildiyse silmeden güncelle
    !git -C /content/proje fetch -q origin && git -C /content/proje reset -q --hard origin/main
else:
    !git clone -q {REPO} /content/proje
assert os.path.isfile("/content/proje/scripts/vs_scripted.py"), "Kod indirilemedi: bu hücrenin çıktısını sohbete yapıştır"
os.chdir("/content/proje")
!git -C /content/proje checkout -q a5d92fd   # bu koşunun kod sürümü
!git log --oneline -1
!pip install -q "crazyflow[gpu]==0.4.0" -e .

In [ ]:
# Bütün koşular burada kayıtlı: değerlendirilecek self-play koşusu (experiments/ altındaki klasör adı).
# RUN'a çalıştırmak istediğinin adını yaz. Neyin koşacağını aşağıdaki bayraklar seçer.
RUNS = ["sp-faz1-evstd-s0", "sp-empty-lr1e4-s0", "sp-empty-anchor-s0", "sp-faz1-anchor2-s0"]
RUN = "eval-sp-faz1-anchor2-s0"
RUN = RUN.removeprefix("eval-")  # runs/ altındaki notebook'un adı "eval-<koşu>"
DO_ELO = True        # Elo turnuvası (olasılıklı, çift başına GAMES maç)
GAMES = 256
DO_VS = True         # unutma testi, gürültüsüz
DO_VS_STOCH = True   # unutma testi, olasılıklı (eğitimdeki gibi)
DO_VS_HELDOUT = True # kovalayanlar eğitimde HİÇ görülmemiş rakibe karşı (hafızalı OPEN, open_mem), gürültüsüz.
#                      Çapa rakipli (anchor_blocks) koşularda OPEN eğitimde görüldüğü için dürüst test bu
WORLDS = 4096        # unutma testinde aynı anda oynanan dünya (dünya başına 2 × 20 s)
OUT = "/content/drive/MyDrive/DroneChaseTag/results"
assert os.path.isdir(f"/content/proje/experiments/{RUN}"), f"experiments/{RUN} repoda yok"
print(RUN, "| Elo:", DO_ELO, "| unutma gürültüsüz:", DO_VS, "| unutma olasılıklı:", DO_VS_STOCH)

In [ ]:
import os
os.chdir("/content/proje")
if DO_ELO:
    !python -W ignore scripts/elo.py --run experiments/{RUN} --games {GAMES} 2>&1 | grep -v -E "warp|spec|fromstring"

In [ ]:
import os
os.chdir("/content/proje")
if DO_VS:
    !python -W ignore scripts/vs_scripted.py --run experiments/{RUN} --worlds {WORLDS} 2>&1 | grep -v -E "warp|spec|fromstring"
if DO_VS_STOCH:
    !python -W ignore scripts/vs_scripted.py --run experiments/{RUN} --worlds {WORLDS} --stochastic 2>&1 | grep -v -E "warp|spec|fromstring"
if DO_VS_HELDOUT:
    !python -W ignore scripts/vs_scripted.py --run experiments/{RUN} --worlds {WORLDS} --roles chaser --opp-vs-chaser open_mem 2>&1 | grep -v -E "warp|spec|fromstring"

In [ ]:
# Sonuç: Elo grafiğini göster, Drive'a kopyala, tek zip olarak indir
import glob, os, shutil, zipfile
from IPython.display import Image, display
from google.colab import files
src = f"/content/proje/experiments/{RUN}"
outs = (["elo"] if os.path.exists(f"{src}/elo") else []) + sorted(os.path.basename(p) for p in glob.glob(f"{src}/vs_scripted*.json"))
if os.path.exists(f"{src}/elo/elo.png"):
    display(Image(f"{src}/elo/elo.png"))
os.makedirs(f"{OUT}/{RUN}", exist_ok=True)
zpath = f"/content/eval-{RUN}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in outs:
        if os.path.isdir(f"{src}/{p}"):
            shutil.copytree(f"{src}/{p}", f"{OUT}/{RUN}/{p}", dirs_exist_ok=True)
            for name in os.listdir(f"{src}/{p}"):
                z.write(f"{src}/{p}/{name}", f"{RUN}/{p}/{name}")
        else:
            shutil.copy(f"{src}/{p}", f"{OUT}/{RUN}/{p}")
            z.write(f"{src}/{p}", f"{RUN}/{p}")
print(zpath, outs)
files.download(zpath)